In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Actual vs predicted SoH (leave-one-vehicle-out) — upgraded model

Uses the shared `src/model.py` rate model (curvature features `1/√age` + `100−SoH`, plateau
down-weighting). For each vehicle we train on the *others* and free-run its SoH over observed
months using its actual stress.

In [ ]:
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
import model

m = pd.read_parquet('data/mahindra/features/feature_table.parquet').sort_values(['vin','month'])
tr_all = model.build_transitions(m)
traj = {}; A = []; P = []
for vin, g in m.groupby('vin'):
    if len(g) < 6:
        continue
    med = model.train_quantiles(tr_all[tr_all['vin'] != vin], alphas=(0.5,))[0.5]
    p = model.free_run_observed(g, med); traj[vin] = (g.sort_values('month'), p)
    A += list(g.sort_values('month')['soh'].values); P += list(p)
A = np.array(A); P = np.array(P)
print(f'LOO over {len(traj)} vehicles | free-run SoH: MAE {mean_absolute_error(A,P):.2f} pp, R2 {r2_score(A,P):.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(A, P, s=8, alpha=0.3); lim = [min(A.min(), P.min())-2, 101]
ax.plot(lim, lim, 'k--', lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('actual SoH (%)'); ax.set_ylabel('predicted SoH (%) — LOO free-run')
ax.set_title(f'Actual vs predicted SoH (all observed months)\nMAE {mean_absolute_error(A,P):.2f} pp, R2 {r2_score(A,P):.2f}')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
deg = [v for v,(g,p) in traj.items() if g['soh'].max()>=93 and g['soh'].min()<=88]
deg = sorted(deg, key=lambda v: -traj[v][0]['age_months'].max())
n=len(deg); ncol=min(4,n) if n else 1; nrow=int(np.ceil(n/ncol)) if n else 1
fig,axes=plt.subplots(nrow,ncol,figsize=(4.4*ncol,3.1*nrow),squeeze=False)
for ax in axes.flat: ax.axis('off')
for i,vin in enumerate(deg):
    g,p=traj[vin]; ax=axes[i//ncol][i%ncol]; ax.axis('on')
    ax.plot(g['month'],g['soh'],'o-',color='C0',ms=3,lw=1.2,label='actual')
    ax.plot(g['month'],p,'s--',color='C1',ms=3,lw=1.2,label='predicted (LOO)')
    ax.set_title(f"{vin[-8:]}  MAE {np.mean(np.abs(g['soh'].values-p)):.1f}pp",fontsize=8)
    ax.grid(alpha=0.3); ax.tick_params(labelsize=7)
    if i==0: ax.legend(fontsize=7)
fig.suptitle('Actual vs predicted SoH (LOO free-run, upgraded model) — degraded vehicles',fontsize=12)
plt.tight_layout(rect=[0,0,1,0.96])
